# XAI for capsule endoscopy: full pipeline on Colab

Runtime > Change runtime type > **T4 GPU** (or better). The dataset (about 520 MB zip,
660 MB of images) goes on the Colab local disk, which is fast. Checkpoints and results
are written to Google Drive so a disconnect does not lose finished work.

Order: setup, data, train both folds, classification and calibration, explanation
localisation, sanity check, copy results to Drive.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/palniraj/xai-capsule-endoscopy.git'  # edit if the repo name differs
DRIVE_DIR = '/content/drive/MyDrive/xai-capsule-endoscopy'           # runs/ and a copy of results/
ARCH = 'resnet50'

In [ ]:
import os
if not os.path.exists('/content/xai-capsule-endoscopy'):
    !git clone -q {REPO_URL} /content/xai-capsule-endoscopy
%cd /content/xai-capsule-endoscopy
!pip -q install captum
# Keep checkpoints on Drive: runs/ is a symlink into DRIVE_DIR.
os.makedirs(f'{DRIVE_DIR}/runs', exist_ok=True)
if not os.path.islink('runs'):
    !rm -rf runs && ln -s {DRIVE_DIR}/runs runs
!ls -la runs

## 1. Data
Downloads the images, metadata and official splits, checks each against a pinned SHA-256,
and writes `data/index.csv` and `results/data_audit.json`. Takes a few minutes.

In [ ]:
!python scripts/prepare_data.py

## 2. Train both directions of the official 2-fold protocol
Each run saves `best.pt` (best validation MCC) and test logits under `runs/`.
Skip a cell if its run already finished (check `runs/<arch>_fold<k>/logits_test.npz`).

In [ ]:
!python scripts/train.py --arch {ARCH} --train-fold 0 --epochs 10 --batch-size 64 --workers 2

In [ ]:
!python scripts/train.py --arch {ARCH} --train-fold 1 --epochs 10 --batch-size 64 --workers 2

## 3. Classification and calibration

In [ ]:
!python scripts/evaluate.py runs/{ARCH}_fold0 runs/{ARCH}_fold1

## 4. Do the explanations point at the lesion?
All boxed lesion frames of the test split, Grad-CAM, Integrated Gradients, Saliency,
and the random and centre reference maps.

In [ ]:
!python scripts/explain_eval.py --run runs/{ARCH}_fold0
!python scripts/explain_eval.py --run runs/{ARCH}_fold1

## 5. Sanity check (cascading model-parameter randomisation)

In [ ]:
!python scripts/sanity_check.py --run runs/{ARCH}_fold0 --n-images 100
!python scripts/sanity_check.py --run runs/{ARCH}_fold1 --n-images 100

## 6. Copy results and figures to Drive
Download `results/` and `figures/` from Drive into the local repo and commit them.

In [ ]:
!mkdir -p {DRIVE_DIR}
!cp -r results figures {DRIVE_DIR}/
!ls {DRIVE_DIR}/results {DRIVE_DIR}/figures

In [ ]:
from IPython.display import Image, display
for f in [f'figures/explanations_{ARCH}_fold0.png', f'figures/sanity_{ARCH}_fold0.png',
          f'figures/reliability_{ARCH}_fold0.png', f'figures/confusion_{ARCH}_fold0.png']:
    display(Image(f))